# Lab 3: Biological Data and Databases, and Using Libraries and Environments.

This lab introduces biological sequence files, Biopython and APIs for querying NCBI and other public bioinformatics resources.

## Learning goals
- Learn to use linux machine and environments created for bioinformatics common analysis (e.g., functions through Biopython, APIs)
- Parse and validate FASTA records with Biopython.
- Retrieve metadata and sequences through NCBI Entrez and REST APIs.
- Understanding scoring matrices (PAM and BLOSUM) and pairwise alignments

## Bio2 environment created for this class
We created an environment (bio2) on Luddy silo machine for this class. The environment was created using conda under /nobackup/yye/Bio2/miniconda3/envs. The environment is meant to be shared (to avoid redundant copies and to be consistent) so students just need to know how to apply the environment. We may need create additional environments for more specific data analysis (e.g., one environment for RNA-seq data analysis) later on. 

### Use Bio2 environment
- Step 1: To use the Bio2 environment on silo, you will have to remote connect to silo via VS Code (or other ways)

- Step 2: Clone this notebook to your folder on silo. 

- Step 3: Setup VSCode to use the environment (choose the python included in the bio2 environment as the python interpreter). It is very likely that VS Code cannot find the python automatically. If it happens, bring up the Command Palette (click the Manage logo, the "wheel"), select Python: Select Interpreter (a Python logo may show up in the left panel of VS Code UI) ![Python Logo](https://github.com/mgtools/Bio2/blob/main/labs/VSCode-logos.png), choose the corresponding directory, and finally set /nobackup/yye/Bio2/miniconda3/envs/bio2/bin/python as the default one. The python for the bio2 will show up as the active python interpreter.

### For people who love command lines
- conda init /nobackup/yye/Bio2/miniconda3/bin/conda
- conda activate bio2

### Plan C -- if nothing works, go with google colab

In [3]:
from io import StringIO
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction
import pandas as pd

fasta = StringIO('>demo_gene_1 organism=example\nATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG\n>demo_gene_2 organism=example\nATGCCTAGGCTAACGTTAGC\n')
records = list(SeqIO.parse(fasta, 'fasta'))
summary = pd.DataFrame([{
    'id': r.id, 'length': len(r.seq), 'gc_percent': round(100 * gc_fraction(r.seq), 2),
    'description': r.description
} for r in records])
summary

ModuleNotFoundError: No module named 'Bio'

In [6]:
from Bio import Entrez
import os

Entrez.email = os.environ.get('NCBI_EMAIL', 'student@example.edu')
term = 'Escherichia coli[Organism] AND lacZ[Gene Name]'
with Entrez.esearch(db='gene', term=term, retmax=5) as handle:
    search = Entrez.read(handle)
search['IdList']

ModuleNotFoundError: No module named 'Bio'

In [ ]:
# Fetch a small, public record. Respect NCBI usage policies and cache larger downloads.
accession = 'NC_000913.3'
with Entrez.efetch(db='nuccore', id=accession, rettype='fasta', retmode='text', strand=1, seq_start=1, seq_stop=500) as handle:
    record = SeqIO.read(handle, 'fasta')
print(record.id, len(record.seq), record.seq[:60])

NC_000913.3:1-500 500 AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTC


## The UniProt REST API provides protein metadata and sequences.


In [ ]:
import requests
response = requests.get(
    'https://rest.uniprot.org/uniprotkb/P69905.fasta',
    headers={'Accept': 'text/plain'}, timeout=30
)
response.raise_for_status()
print(response.text[:200])

>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 GN=HBA1 PE=1 SV=2
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG
KKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLA


In [ ]:
## SeqKit workflow
Save a FASTA file and run these commands in a terminal. Compare their output with the Python summary above.

```bash
seqkit stats data/example.fasta
seqkit fx2tab -n -l -g data/example.fasta
seqkit grep -s -r -p 'ATG' data/example.fasta
```

For your report, explain when a command-line tool is preferable to loading all records into memory.

SyntaxError: invalid syntax (1093368520.py, line 2)

## Exercises
1. Download a FASTA file for a species of your choice and produce a table of lengths and GC percentages.
2. Use SeqKit to filter records by length and motif, then reproduce the result in Biopython.
3. Query NCBI for three genes, retrieve their FASTA records, and save a provenance table containing database, query, date, and accession.
4. Choose one API from UniProt, Europe PMC, or ENA. Document its endpoint, parameters, response format, and one limitation.

**Deliverable:** notebook outputs, commands used, and a short paragraph about reproducibility and responsible API use.